**DETECCIÓN Y LECTURA DE MATRÍCULAS EN VÍDEO CON RT-DETRv2 Y PADDLEOCR-VL, UN VLM**

En este cuaderno se detectan personas y vehículos con RT-DETRv2, se les asigna un identificador con ByteTrack, se localiza la matrícula de cada vehículo con el detector de matrículas (RT-DETRv2 tamaño `s`) y se lee su texto con el modelo de visión-lenguaje `PaddlePaddle/PaddleOCR-VL`.

El flujo de trabajo se estructura en:
1. **Configuración inicial**: Importaciones, clonado de RT-DETR y selección del dispositivo (GPU si está disponible).
2. **Modelo de matrículas**: Carga del detector de matrículas.
3. **Funciones de OCR con VLM**: Carga de PaddleOCR-VL y lectura de matrículas con su confianza.
4. **Parámetros del vídeo y modelo de vehículos**: Parámetros de detección, seguimiento y OCR, y carga del detector de vehículos.
5. **Procesamiento de vídeo**: Detección, seguimiento, localización de matrículas y OCR sobre un vídeo, guardando el vídeo anotado y un CSV.
6. **Evaluación del VLM**: Exact match y acierto por carácter sobre imágenes anotadas, con y sin corrección por posición, y tiempos de lectura.

**CONFIGURACIÓN INICIAL: IMPORTS Y DISPOSITIVO**

Se clona el repositorio de RT-DETR si no existe, se añade su ruta a `sys.path`, se define el identificador del VLM y se selecciona la GPU si PyTorch detecta una con CUDA.

In [1]:
import os, sys, subprocess
import csv
from functools import partial
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#Cargamos el modelo y ponemos la ruta en memoria
if not os.path.exists("RT-DETR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lyuwenyu/RT-DETR.git"], check=True)


RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")
if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from load_rtdetr import load_rtdetr
from utils_matriculas import (
    GT_KEYS,
    cargar_gt,
    get_ocr_vlm,
    iou,
    leer_cronometrado,
    leer_placa_vlm,
    letterbox,
    lev,
    procesar_video_rtdetr,
    recortar,
    run_detector,
)

VLM_ID = "PaddlePaddle/PaddleOCR-VL"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\enriq\anaconda3\envs\matriculas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**MODELO DE MATRÍCULAS**

Se muestran las configuraciones disponibles en el repositorio y se carga el detector de matrículas (RT-DETRv2 tamaño `s`), con una única clase (`plate`), que se aplica sobre el recorte de cada vehículo.

In [2]:
repo = Path("RT-DETR/rtdetrv2_pytorch")

print("Configuraciones RT-DETRv2:")
for f in (repo / "configs/rtdetrv2").glob("*.yml"):
    print(f.name)

print("\nConfiguraciones de matrículas en todo el repositorio:")
for f in repo.rglob("*plate*.yml"):
    print(f)

PLATE_NAME = {0: "matrícula"}
PLATE_CLASS = PLATE_NAME.keys()
PLATE_COLOR = {0: (128, 255, 128)}

model_plate, postprocessor_plate = load_rtdetr(family="v2", size="s", device=DEVICE)

Configuraciones RT-DETRv2:
rtdetrv2_hgnetv2_h_6x_coco.yml
rtdetrv2_hgnetv2_l_6x_coco.yml
rtdetrv2_hgnetv2_x_6x_coco.yml
rtdetrv2_r101vd_6x_coco.yml
rtdetrv2_r18vd_120e_coco.yml
rtdetrv2_r18vd_120e_plates.yml
rtdetrv2_r18vd_120e_voc.yml
rtdetrv2_r18vd_dsp_3x_coco.yml
rtdetrv2_r18vd_sp1_120e_coco.yml
rtdetrv2_r18vd_sp2_120e_coco.yml
rtdetrv2_r18vd_sp3_120e_coco.yml
rtdetrv2_r34vd_120e_coco.yml
rtdetrv2_r34vd_dsp_1x_coco.yml
rtdetrv2_r50vd_6x_coco.yml
rtdetrv2_r50vd_dsp_1x_coco.yml
rtdetrv2_r50vd_m_7x_coco.yml
rtdetrv2_r50vd_m_dsp_3x_coco.yml

Configuraciones de matrículas en todo el repositorio:
RT-DETR\rtdetrv2_pytorch\configs\dataset\plate_detection.yml
RT-DETR\rtdetrv2_pytorch\configs\rtdetrv2\rtdetrv2_r18vd_120e_plates.yml
Cargando RT-DETR familia 'v2', tamaño 's' en cuda


**FUNCIONES DE OCR: PADDLEOCR-VL**

Se carga el VLM una sola vez `get_ocr_vlm` y se fija la función de lectura `leer_placa`.
- **`norm`**: Normaliza el texto a mayúsculas alfanuméricas.
- **`recortar`**: Recorta una región con margen porcentual o en píxeles, ajustada a los límites de la imagen.
- **`leer_placa_vlm`**: Reescala el recorte, lo pasa al VLM con el prompt `OCR:` y devuelve el texto y la confianza media de los tokens generados.

Aquí se fijan con `partial` el dispositivo y el modelo ya cargado, para que `leer_placa(crop)` reciba solo el recorte.

En `get_ocr_vlm` el tipo de dato se elige según `DEVICE.type`: `bfloat16` en CUDA y `float32` en CPU. Si `bfloat16` diera un error de tipos en tu GPU, cambia `torch.bfloat16` por `torch.float32`.

In [3]:
# Inicializamos el VLM globalmente: devuelve (modelo, processor), o None si falla la carga
ocr_global = get_ocr_vlm(DEVICE, VLM_ID)

# leer_placa_vlm (utils) recibe el device y el modelo como argumentos: los fijamos aqui
# para conservar la firma leer_placa(crop) -> (texto, confianza)
leer_placa = partial(leer_placa_vlm, device=DEVICE, ocr_inst=ocr_global)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


**PARÁMETROS DEL VÍDEO Y MODELO DE VEHÍCULOS**

Parámetros de detección, seguimiento y OCR sobre vídeo:
- **Clases y colores**: Personas, coches, motos, autobuses y camiones.
- **Recortes de vehículo**: Las matrículas se buscan dentro del recorte de cada vehículo y heredan su `tracker_id`.
- **Lectura OCR**: Frecuencia, número máximo de intentos y confianza mínima para dar una lectura por buena.
- **Patrones de matrícula**: Las expresiones regulares de formatos europeos `PATRONES_MATRICULA` y `matricula_valida`.

Aquí se carga también el detector de vehículos y personas (RT-DETRv2 tamaño `l`).

In [4]:
CLASS_NAMES = {0: 'persona', 2: 'coche', 3: 'moto', 5: 'guagua', 7: 'camion'} 
BASE_CLASS = CLASS_NAMES.keys()
BASE_COLORS = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255), 5: (0, 255, 255), 7: (128, 128, 255)}

# Las matriculas se buscan dentro del recorte de cada vehiculo y heredan su tracker_id
MAX_CROPS = 8          # maximo de vehiculos por frame donde se buscan matriculas
MIN_CROP_SIDE = 48     # px: vehiculos mas pequenos que esto se ignoran
CROP_MARGIN = 0.05     # margen extra alrededor del vehiculo
PLATE_CONF = 0.3      # confianza minima del detector de matriculas
HOLD_FRAMES = 5        # frames que se mantiene la caja de matricula si no se detecta

# Parámetros visuales y de OCR de matrículas
COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

model, postprocessor = load_rtdetr(family="v2", size="l", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 'l' en cuda
Load PResNet50 state_dict


**TRACKING Y DETECCIÓN RT-DETR**

Las funciones del procesamiento de vídeo están en `utils_matriculas.py`:
- **`preprocess`**: Redimensiona el fotograma a 640x640 y lo convierte en tensor, una sola vez por fotograma.
- **`make_tracker`**: Crea un tracker ByteTrack con el tiempo de memoria indicado.
- **`letterbox`**: Redimensiona el recorte manteniendo la proporción y rellena con gris, para no deformar las matrículas.
- **`run_detector`**: Ejecuta el detector y filtra por confianza y clases.
- **`detect_plates_in_crops`**: Busca la mejor matrícula dentro del recorte de cada vehículo y la devuelve en coordenadas del fotograma.

**PROCESAMIENTO DE VÍDEO**

Se recorre el vídeo fotograma a fotograma: se detectan personas y vehículos, se les asigna un identificador, se busca la matrícula de cada vehículo y se lee con el VLM. Se guardan el vídeo anotado (`resultado.mp4`) y un CSV con todos los resultados (`resultado.csv`) en `RT-DETR_VLM_result`.

In [5]:
path_test = 'C0142.MP4'
output_dir = os.path.abspath('RT-DETR_VLM_result')

plates_seen, lecturas = procesar_video_rtdetr(
    path_test, output_dir, model, postprocessor, model_plate, postprocessor_plate,
    DEVICE, leer_placa, CLASS_NAMES, BASE_COLORS,
    plate_classes=PLATE_CLASS,
    conf_vehiculos=0.5, 
    plate_conf=PLATE_CONF, hold_frames=HOLD_FRAMES, max_crops=MAX_CROPS,
    min_crop_side=MIN_CROP_SIDE, crop_margin=CROP_MARGIN,
    color_placa=COLOR_PLACA, escala_texto_placa=ESCALA_TEXTO_PLACA,
    ocr_cada=OCR_CADA, min_ancho_ocr=MIN_ANCHO_OCR, min_chars_placa=MIN_CHARS_PLACA,
    ocr_score_ok=OCR_SCORE_OK, ocr_max_intentos=OCR_MAX_INTENTOS, vlm=True
)

Video guardado en c:\Users\enriq\Desktop\VC\Practicas_VC\P4\RT-DETR_VLM_result\resultado.mp4 | CSV guardado en c:\Users\enriq\Desktop\VC\Practicas_VC\P4\RT-DETR_VLM_result\resultado.csv
Matrículas detectadas: 47 | Matrículas leídas con OCR: 14


**EVALUACIÓN DEL VLM SOBRE EL DATASET**

Se evalúa el conjunto detector de matrículas + VLM sobre las imágenes anotadas, comparando contra el texto real. Se obtienen:
- **Exact match**: Proporción de matrículas leídas exactamente igual que la anotación.
- **Acierto por carácter**: Similitud media basada en la distancia de Levenshtein.
- **Techo del OCR**: Lectura directa del recorte de la anotación, sin depender del detector.
- **Lectura cruda vs. corregida**: El VLM tal cual y tras la corrección por posición.
- **Tiempos**: Milisegundos por lectura (media y mediana), con una lectura de calentamiento previa.

**Configuración**: rutas del dataset, umbrales, carpetas de salida y modelo VLM propio de esta evaluación.

In [ ]:
# ---------- Modelo VLM ----------
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO = torch.bfloat16 if DISPOSITIVO == "cuda" else torch.float32
ID_MODELO = "PaddlePaddle/PaddleOCR-VL"

vlm = AutoModelForCausalLM.from_pretrained(
    ID_MODELO, trust_remote_code=True, torch_dtype=TIPO).to(DISPOSITIVO).eval()
procesador = AutoProcessor.from_pretrained(ID_MODELO, trust_remote_code=True)

# ---------- Configuración de la evaluación ----------
BASE = Path("datos")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"

MAX_IMAGENES = 200
CONF_EVAL = 0.25
IOU_MATCH = 0.5
PAD = 0.05
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")
DIR_ERRORES = SALIDA / "errores"
DIR_ERRORES.mkdir(parents=True, exist_ok=True)

# Tensor constante para el detector (se crea una sola vez)
TAM_DET = torch.tensor([[640, 640]], device=DISPOSITIVO)

**UTILIDADES DE LA EVALUACIÓN**

- **`gt_dataset`**: `cargar_gt` con la carpeta de anotaciones y las claves ya fijadas.
- **`detectar_rtdetr`**: Detecta matrículas con RT-DETR sobre la imagen completa y devuelve las cajas en píxeles originales.
- **`fila`**, **`resumen`**, **`texto_valido`**: Construcción de filas del CSV, cálculo de métricas y comprobación de que la anotación tiene texto.

In [ ]:
# ---------- Utilidades ----------
# cargar_gt (utils) recibe la carpeta de anotaciones y las claves como argumentos: las fijamos aqui
gt_dataset = partial(cargar_gt, anotaciones_dir=ANOTACIONES, gt_keys=GT_KEYS)


@torch.no_grad()
def detectar_rtdetr(img):
    """Detecta matrículas con RT-DETR y devuelve cajas en píxeles originales."""
    h, w = img.shape[:2]

    # Preprocesamiento igual que en el detector RT-DETR
    x, r, px, py = letterbox(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), DEVICE)
    cajas, scores, _ = run_detector(x, model_plate, postprocessor_plate,
                                    CONF_EVAL, list(PLATE_CLASS), TAM_DET)[0]

    preds = []
    for (a, b, c, d), conf in zip(cajas, scores):
        # Deshacer letterbox y limitar a los bordes de la imagen
        x1, x2 = (a - px) / r, (c - px) / r
        y1, y2 = (b - py) / r, (d - py) / r
        preds.append(([max(0, min(w, x1)), max(0, min(h, y1)),
                       max(0, min(w, x2)), max(0, min(h, y2))], float(conf)))
    return preds


def fila(imagen, origen, gt, cruda, corregida, ms, det_conf, iou_v):
    return dict(imagen=imagen, origen=origen, gt=gt, ocr_crudo=cruda, ocr_corregido=corregida,
                ms=round(ms, 1), det_conf=det_conf, iou=iou_v)


def resumen(titulo, rows, col):
    """Calcula acierto exacto, por carácter y tiempos."""
    if not rows:
        print(f"{titulo}: sin resultados")
        return

    exacto = np.mean([r["gt"] == r[col] for r in rows])
    por_car = np.mean([1 - min(1, lev(r["gt"], r[col]) / max(len(r["gt"]), 1)) for r in rows])
    tiempos = [r["ms"] for r in rows]

    print(f"{titulo:<40} n={len(rows):<5} exacto={exacto:.3f} | por carácter={por_car:.3f} | "
          f"{np.mean(tiempos):.0f} ms de media ({np.median(tiempos):.0f} de mediana)")


def texto_valido(t):
    return t is not None and t != "NONE"

**EJECUCIÓN DE LA EVALUACIÓN**

Para cada imagen se hace OCR sobre las detecciones de RT-DETR y sobre el recorte exacto de la anotación. Se guardan el CSV `rtdetr_resultados_vlm.csv` y los recortes mal leídos en `vlm_eval/errores`, y se imprime el resumen.

In [ ]:
# ---------- Evaluación ----------
leer_cronometrado(np.zeros((30, 100, 3), np.uint8), leer_placa)  # calentamiento del OCR

lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
if MAX_IMAGENES:
    lista = lista[:MAX_IMAGENES]

filas = []
n_gt = n_gt_texto = n_detectadas = 0

for ruta_img in lista:
    img = cv2.imread(str(ruta_img))
    if img is None:
        continue

    h_img, w_img = img.shape[:2]
    gts = gt_dataset(ruta_img, w_img, h_img)
    n_gt += len(gts)
    n_gt_texto += sum(t is not None for _, t in gts)

    preds = detectar_rtdetr(img)
    n_detectadas += len(preds)

    # 1) OCR sobre las detecciones de RT-DETR (solo si hay anotación con la que compararlas)
    for caja, conf_det in (preds if gts else ()):
        # Anotación con mayor IoU (la primera en caso de empate)
        mejor_iou, gt_txt = 0.0, None
        for gcaja, gtxt in gts:
            v = iou(caja, gcaja)
            if v > mejor_iou:
                mejor_iou, gt_txt = v, gtxt

        # Descarta antes de hacer OCR (era lo más costoso y se tiraba el resultado)
        if mejor_iou < IOU_MATCH or not texto_valido(gt_txt):
            continue

        recorte = recortar(img, caja, pad=PAD)
        cruda, corregida, ms = leer_cronometrado(recorte, leer_placa)
        filas.append(fila(ruta_img.name, "deteccion", gt_txt, cruda, corregida, ms,
                          round(conf_det, 3), round(mejor_iou, 3)))

        if gt_txt != cruda:
            cv2.imwrite(str(DIR_ERRORES / f"{len(filas):05d}_gt-{gt_txt}_ocr-{cruda}.jpg"), recorte)

    # 2) OCR sobre el recorte exacto de la anotación (mide el OCR sin depender del detector)
    for gcaja, gtxt in gts:
        if not texto_valido(gtxt):
            continue
        cruda, corregida, ms = leer_cronometrado(recortar(img, gcaja, pad=PAD), leer_placa)
        filas.append(fila(ruta_img.name, "gt_recorte", gtxt, cruda, corregida, ms, None, 1.0))


# ---------- Guardar y resumir ----------
ruta_csv = SALIDA / "rtdetr_resultados_vlm.csv"

with open(ruta_csv, "w", newline="", encoding="utf-8") as f:
    escritor_csv = csv.DictWriter(f, fieldnames=list(filas[0]) if filas else ["vacio"])
    escritor_csv.writeheader()
    escritor_csv.writerows(filas)

print(f"\nImágenes: {len(lista)} | placas anotadas: {n_gt} | detecciones RT-DETR: {n_detectadas}")

if n_gt_texto == 0:
    print(f"Las anotaciones no traen texto (claves probadas: {GT_KEYS}).")
else:
    det = [r for r in filas if r["origen"] == "deteccion"]
    techo = [r for r in filas if r["origen"] == "gt_recorte"]

    resumen("VLM sobre detecciones RT-DETR", det, "ocr_crudo")
    resumen("VLM sobre recorte anotado (techo)", techo, "ocr_crudo")
    resumen("VLM + corrección, detecciones", det, "ocr_corregido")
    resumen("VLM + corrección, recorte anotado", techo, "ocr_corregido")

    print(f"Placas anotadas encontradas por RT-DETR: {len(det)}/{n_gt_texto}")

print(f"\nCSV: {ruta_csv}")
print(f"Recortes con error: {DIR_ERRORES}")